In [1]:
from pypdf import PdfReader
import os


pdf_path = r"D:\ITI_applied_ai\Graduation Project\united_nations_charter 1.pdf"

if os.path.exists(pdf_path):
    reader = PdfReader(pdf_path)
    num_pages = len(reader.pages)

    print(f"Total pages: {num_pages}")
    print("-" * 40)
    
    
    
    sample_text = reader.pages[0].extract_text()
    
    print("--- Sample text from Page 1 ---")
    print(sample_text[:512])
else:
    print(f"File not found! Please check the path: {pdf_path}")

Total pages: 23
----------------------------------------
--- Sample text from Page 1 ---
 
CHAPTER I: PURPOSES AND PRINCIPLES 
 
Article 1 
The Purposes of the United Nations are: 
 
1. To maintain international peace and security, and to that end: to take effective collective 
measures for the prevention and removal of threats to the peace, and for the suppression of 
acts of aggression or other breaches of the peace, and to bring about by peaceful means, and 
in conformity with the principles of justice and international law, adjustment or settlement of 
international disputes or situations w


In [2]:
import re
from pypdf import PdfReader

pdf_path = r"D:\ITI_applied_ai\Graduation Project\united_nations_charter 1.pdf"
reader = PdfReader(pdf_path)

current_chapter = "N/A"
current_article = "N/A"
current_text_buffer = []

# ضفنا المتتبع ده عشان نتأكد إن المادة ماشية بالترتيب
expected_article_num = 1 

chapter_pattern = re.compile(r"^CHAPTER\s+[IVXLCDM]+", re.IGNORECASE)
# ضفنا أقواس () حوالين \d+ عشان نقدر نستخرج الرقم كقيمة ونختبره
article_pattern = re.compile(r"^Article\s+(\d+)", re.IGNORECASE)

documents_data = []

def save_current_chunk(page_num):
    text = " ".join(current_text_buffer).strip()
    text = re.sub(r'\s+', ' ', text)
    
    if text:
        documents_data.append({
            "text": text,
            "metadata": {
                "page": page_num + 1,
                "chapter": current_chapter,
                "article": current_article
            }
        })
    current_text_buffer.clear()

for page_num, page in enumerate(reader.pages):
    text = page.extract_text()
    if not text:
        continue
        
    for line in text.split('\n'):
        line = line.strip()
        if not line:
            continue
            
        # 1. اكتشاف فصل جديد
        chapter_match = chapter_pattern.match(line)
        if chapter_match:
            save_current_chunk(page_num)
            current_chapter = chapter_match.group()
            current_article = "N/A" 
            
            remainder = line[chapter_match.end():].strip()
            if remainder:
                current_chapter += " " + remainder
            continue
            
        # 2. استغلال الـ UPPERCASE
        if line.isupper() and current_article == "N/A" and current_chapter != "N/A":
            current_chapter += f" {line}"
            continue
                
        # 3. اكتشاف مادة جديدة (هنا سحر التسلسل)
        article_match = article_pattern.match(line)
        if article_match:
            found_num = int(article_match.group(1)) # الرقم اللي لقيناه في السطر
            
            # هل ده الرقم اللي عليه الدور فعلاً؟
            if found_num == expected_article_num:
                save_current_chunk(page_num)
                current_article = f"Article {found_num}"
                expected_article_num += 1 # زود العداد للمادة اللي بعدها
                
                remainder = line[article_match.end():].strip()
                if remainder:
                    current_text_buffer.append(remainder)
                continue
            else:
                # لو مش هو الرقم اللي عليه الدور (زي Article 10 جوه Article 12)
                # الكود هيتجاهل الشرط ده تماماً وينزل يضيفه كنص عادي في الخزان!
                pass
            
        # 4. إضافة السطر العادي
        current_text_buffer.append(line)

save_current_chunk(page_num)

print(f"Total chunks created: {len(documents_data)}\n")

Total chunks created: 111



In [3]:
# حساب أطوال كل الـ Chunks
chunk_lengths = [len(doc['text']) for doc in documents_data]

max_length = max(chunk_lengths)
min_length = min(chunk_lengths)
avg_length = sum(chunk_lengths) / len(chunk_lengths)

# إيجاد بيانات أكبر Chunk لنتعرف عليه
largest_chunk = max(documents_data, key=lambda x: len(x['text']))

print(f"Max Chunk Size: {max_length} characters")
print(f"Min Chunk Size: {min_length} characters")
print(f"Average Chunk Size: {avg_length:.0f} characters")
print("-" * 50)
print(f"The largest chunk is in: {largest_chunk['metadata']['chapter']} | {largest_chunk['metadata']['article']}")

Max Chunk Size: 1759 characters
Min Chunk Size: 107 characters
Average Chunk Size: 469 characters
--------------------------------------------------
The largest chunk is in: CHAPTER XI : DECLARATION REGARDING NON-SELF- GOVERNING TERRITORIES | Article 73


In [4]:
from transformers import AutoTokenizer

# 1. تحميل الـ Tokenizer الخاص بنفس النموذج الذي سنستخدمه
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# 2. حساب عدد الـ Tokens لكل Chunk وحفظه داخل القاموس
for doc in documents_data:
    # نقوم بتمرير النص للـ tokenizer ليخبرنا بعدد الرموز (Tokens)
    tokens = tokenizer.encode(doc['text'])
    doc['token_count'] = len(tokens)

# 3. ترتيب البيانات تصاعدياً بناءً على عدد الـ Tokens (من الأصغر للأكبر)
sorted_docs = sorted(documents_data, key=lambda x: x['token_count'])

# 4. طباعة أصغر 20 Chunk لاكتشاف أي أخطاء
print("--- Top 20 Smallest Chunks (By Token Count) ---\n")
for i in range(min(20, len(sorted_docs))):
    doc = sorted_docs[i]
    print(f"Rank: {i+1}")
    print(f"Tokens: {doc['token_count']} | Characters: {len(doc['text'])}")
    print(f"Metadata: {doc['metadata']}")
    print(f"Text: '{doc['text']}'")
    print("-" * 50)

# 5. طباعة معلومات أكبر Chunk بالـ Tokens للتأكد من الأمان
largest_token_chunk = max(documents_data, key=lambda x: x['token_count'])
print(f"\nMax Tokens in a single chunk: {largest_token_chunk['token_count']}")

--- Top 20 Smallest Chunks (By Token Count) ---

Rank: 1
Tokens: 21 | Characters: 109
Metadata: {'page': 7, 'chapter': 'CHAPTER V : THE SECURITY COUNCIL', 'article': 'Article 30'}
Text: 'The Security Council shall adopt its own rules of procedure, including the method of selecting its President.'
--------------------------------------------------
Rank: 2
Tokens: 22 | Characters: 107
Metadata: {'page': 6, 'chapter': 'CHAPTER IV : THE GENERAL ASSEMBLY', 'article': 'Article 21'}
Text: 'The General Assembly shall adopt its own rules of procedure. It shall elect its President for each session.'
--------------------------------------------------
Rank: 3
Tokens: 22 | Characters: 117
Metadata: {'page': 6, 'chapter': 'CHAPTER IV : THE GENERAL ASSEMBLY', 'article': 'Article 22'}
Text: 'The General Assembly may establish such subsidiary organs as it deems necessary for the performance of its functions.'
--------------------------------------------------
Rank: 4
Tokens: 22 | Characters: 117
Metada

In [5]:
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. التأكد أنك عندك documents_data جاهزة من الكود السابق
print(f"Total chunks ready to be stored: {len(documents_data)}")

# 2. تحديد مسار حفظ قاعدة البيانات محلياً
db_path = r"D:\ITI_applied_ai\Graduation Project\rag-assistant-project"
os.makedirs(db_path, exist_ok=True)

# 3. تشغيل ChromaDB بوضع الـ Persistent (حفظ على القرص)
chroma_client = chromadb.PersistentClient(path=db_path)

# 4. اختيار نموذج الـ Embedding (all-MiniLM-L6-v2)
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# 5. إنشاء أو ربط الـ Collection
collection = chroma_client.get_or_create_collection(
    name="un_charter_collection",
    embedding_function=sentence_transformer_ef
)

# 6. تجهيز الأجزاء وإدخالها (ChromaDB هيعمل الـ Embedding أوتوماتيك للنصوص اللي طلعتها)
ids = [f"chunk_{idx}" for idx in range(len(documents_data))]
documents = [doc['text'] for doc in documents_data]
metadatas = [doc['metadata'] for doc in documents_data]

print("Generating embeddings and saving to ChromaDB...")
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print(f"Successfully stored {collection.count()} chunks in ChromaDB at path: {db_path}")

Total chunks ready to be stored: 111


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings and saving to ChromaDB...
Successfully stored 111 chunks in ChromaDB at path: D:\ITI_applied_ai\Graduation Project\rag-assistant-project


In [6]:
import os
print("Current Working Directory:", os.getcwd())

Current Working Directory: D:\ITI_applied_ai\Graduation Project\rag-assistant-project


In [7]:
import json
import os

config_path = r"D:\ITI_applied_ai\Graduation Project\rag-assistant-project\config.json"

config_data = {
    "embedding_model": "all-MiniLM-L6-v2",
    "vector_database": "ChromaDB",
    "total_chunks": 111,
    "chunking_strategy": "Article-level semantic chunking with regex filtering"
}


with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=4)

print(f"Configuration file successfully saved at: {config_path}")

Configuration file successfully saved at: D:\ITI_applied_ai\Graduation Project\rag-assistant-project\config.json


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


def embed_query(query):
    """
    Convert the user's query into an embedding vector.
    """
    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True
    )

    return query_embedding

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
query = "What are the purposes of the United Nations?"

query_embedding = embed_query(query)

print("Embedding shape:", query_embedding.shape)
print("First 5 values:", query_embedding[:5])

Embedding shape: (384,)
First 5 values: [-0.06086611  0.09456176 -0.03977093 -0.04992181 -0.00860056]


In [10]:
import chromadb
from chromadb.utils import embedding_functions

# Path of the persisted ChromaDB
db_path = r"D:\ITI_applied_ai\Graduation Project\rag-assistant-project"

# Load existing ChromaDB
chroma_client = chromadb.PersistentClient(path=db_path)

# Same embedding model used during indexing
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Load existing collection
collection = chroma_client.get_collection(
    name="un_charter_collection",
    embedding_function=sentence_transformer_ef
)

print("Collection loaded successfully!")
print("Number of chunks:", collection.count())

Collection loaded successfully!
Number of chunks: 111


In [11]:
import numpy as np 
data = collection.get(
    include=["documents", "metadatas", "embeddings"]
)

embeddings = np.array(data["embeddings"])
documents = data["documents"]
metadatas = data["metadatas"]

In [12]:
print(len(data["embeddings"]))
print(data["embeddings"][0].shape)

111
(384,)


In [13]:
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def get_retrieval_results(
    query,
    query_embedding,
    document_embeddings,
    documents,
    metadatas,
    k=5,
    threshold=0.70
):
    """
    Retrieve relevant chunks using metadata search or
    semantic similarity search.

    Metadata search is used when the query explicitly
    mentions an Article or Chapter.
    """

    # =====================================================
    # 1. Check if query contains an Article number
    # =====================================================

    article_match = re.search(
        r'\barticle\s+(\d+)\b',
        query,
        re.IGNORECASE
    )

    if article_match:

        article_number = article_match.group(1)

        target_article = f"Article {article_number}"

        results = []

        for idx, metadata in enumerate(metadatas):

            if metadata.get("article", "").lower() == target_article.lower():

                results.append({
                    "text": documents[idx],
                    "metadata": metadata,
                    "similarity": "Article"
                })

        return results


    # =====================================================
    # 2. Check if query contains a Chapter
    # =====================================================

    chapter_match = re.search(
        r'\bchapter\s+([IVXLCDM]+|\d+)\b',
        query,
        re.IGNORECASE
    )
    
    if chapter_match:
    
        chapter_number = chapter_match.group(1).upper()
    
        results = []
    
        for idx, metadata in enumerate(metadatas):
    
            stored_chapter = metadata.get("chapter", "")
    
            stored_match = re.search(
                r'\bchapter\s+([IVXLCDM]+|\d+)\b',
                stored_chapter,
                re.IGNORECASE
            )
    
            if stored_match:
    
                stored_chapter_number = stored_match.group(1).upper()
    
                if stored_chapter_number == chapter_number:
    
                    results.append({
                        "text": documents[idx],
                        "metadata": metadata,
                        "similarity": "Chapter"
                    })
    
        return results
    # =====================================================
    # 3. Normal Semantic Search
    # =====================================================

    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        document_embeddings
    )[0]

    # Keep only results >= threshold
    valid_indices = np.where(
        similarities >= threshold
    )[0]

    # Sort from highest similarity to lowest
    sorted_indices = valid_indices[
        np.argsort(similarities[valid_indices])[::-1]
    ]

    # Take top-k
    top_indices = sorted_indices[:k]

    results = []

    for idx in top_indices:

        results.append({
            "text": documents[idx],
            "metadata": metadatas[idx],
            "similarity": float(similarities[idx])
        })

    return results

In [14]:
query = "What does Article 13 say?"

query_embedding = embed_query(query)

results = get_retrieval_results(
    query=query,
    query_embedding=query_embedding,
    document_embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    k=5,
    threshold=0.70
)

In [15]:
for i, result in enumerate(results, 1):

    print(f"\nResult {i}")
    print("=" * 50)

    print(f"Similarity: {result['similarity']}")
    print(f"Metadata: {result['metadata']}")
    print(f"Text:\n{result['text']}")


Result 1
Similarity: Article
Metadata: {'page': 4, 'article': 'Article 13', 'chapter': 'CHAPTER IV : THE GENERAL ASSEMBLY'}
Text:
1. The General Assembly shall initiate studies and make recommendations for the purpose of: a. promoting international co-operation in the political field and encouraging the progressive development of international law and its codification; b. promoting international co-operation in the economic, social, cultural, educational, and health fields, and assisting in the realization of human rights and fundamental freedoms for all without distinction as to race, sex, language, or religion. c. The further responsibilities, functions and powers of the General Assembly with respect to matters mentioned in paragraph 1 (b) above are set forth in Chapters IX and X.


In [16]:
query = "What does Chapter I say?"

query_embedding = embed_query(query)

results = get_retrieval_results(
    query=query,
    query_embedding=query_embedding,
    document_embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    k=5,
    threshold=0.70
)
for i, result in enumerate(results, 1):

    print(f"\nResult {i}")
    print("=" * 50)

    print(f"Similarity: {result['similarity']}")
    print(f"Metadata: {result['metadata']}")
    print(f"Text:\n{result['text']}")



Result 1
Similarity: Chapter
Metadata: {'chapter': 'CHAPTER I : PURPOSES AND PRINCIPLES', 'page': 2, 'article': 'Article 1'}
Text:
The Purposes of the United Nations are: 1. To maintain international peace and security, and to that end: to take effective collective measures for the prevention and removal of threats to the peace, and for the suppression of acts of aggression or other breaches of the peace, and to bring about by peaceful means, and in conformity with the principles of justice and international law, adjustment or settlement of international disputes or situations which might lead to a breach of the peace; 2. To develop friendly relations among nations based on respect for the principle of equal rights and self-determination of peoples, and to take other appropriate measures to strengthen universal peace; 3. To achieve international co-operation in solving international problems of an economic, social, cultural, or humanitarian character, and in promoting and encouraging 

In [17]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [18]:
def rerank_results(query, results):
    """
    Rerank the retrieved chunks using a Cross-Encoder
    and return the best chunk.
    """

    # Create query-chunk pairs
    pairs = [
        [query, result["text"]]
        for result in results
    ]

    # Calculate reranker scores
    scores = reranker.predict(pairs)

    # Add reranker score to each result
    for result, score in zip(results, scores):
        result["reranker_score"] = float(score)

    # Sort by reranker score
    results = sorted(
        results,
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    # Return the best chunk only
    return results[0] if results else None

In [19]:
query = "What are the purposes of the United Nations?"

query_embedding = embed_query(query)

results = get_retrieval_results(
    query=query,
    query_embedding=query_embedding,
    document_embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    k=5,
    threshold=0.70
)
for i, result in enumerate(results, 1):

    print(f"\nResult {i}")
    print("=" * 50)

    print(f"Similarity: {result['similarity']}")
    print(f"Metadata: {result['metadata']}")
    print(f"Text:\n{result['text']}")


Result 1
Similarity: 0.8214173716034494
Metadata: {'chapter': 'CHAPTER I : PURPOSES AND PRINCIPLES', 'page': 2, 'article': 'Article 1'}
Text:
The Purposes of the United Nations are: 1. To maintain international peace and security, and to that end: to take effective collective measures for the prevention and removal of threats to the peace, and for the suppression of acts of aggression or other breaches of the peace, and to bring about by peaceful means, and in conformity with the principles of justice and international law, adjustment or settlement of international disputes or situations which might lead to a breach of the peace; 2. To develop friendly relations among nations based on respect for the principle of equal rights and self-determination of peoples, and to take other appropriate measures to strengthen universal peace; 3. To achieve international co-operation in solving international problems of an economic, social, cultural, or humanitarian character, and in promoting and e

In [20]:
best_chunk = rerank_results(
    query=query,
    results=results
)

In [21]:
print(f"Similarity: {best_chunk['similarity']:}") 
print(f"Reranker Score: {best_chunk['reranker_score']:}") 
print(f"Metadata: {best_chunk['metadata']}") 
print(f"Text:\n{best_chunk['text']}")

Similarity: 0.8214173716034494
Reranker Score: 10.51416015625
Metadata: {'chapter': 'CHAPTER I : PURPOSES AND PRINCIPLES', 'page': 2, 'article': 'Article 1'}
Text:
The Purposes of the United Nations are: 1. To maintain international peace and security, and to that end: to take effective collective measures for the prevention and removal of threats to the peace, and for the suppression of acts of aggression or other breaches of the peace, and to bring about by peaceful means, and in conformity with the principles of justice and international law, adjustment or settlement of international disputes or situations which might lead to a breach of the peace; 2. To develop friendly relations among nations based on respect for the principle of equal rights and self-determination of peoples, and to take other appropriate measures to strengthen universal peace; 3. To achieve international co-operation in solving international problems of an economic, social, cultural, or humanitarian character, a

In [22]:
test_questions = [
    # Chapter-based questions
    "What does Chapter I of the UN Charter say?",
    "What is covered in Chapter V of the UN Charter?",

    # Article-based questions
    "What does Article 46 say?",
    "What are the functions of the General Assembly according to Article 10?",

    # Semantic questions related to the UN Charter
    "What are the purposes of the United Nations?",
    "What principles should the United Nations and its members follow?",
    "What does the UN Charter say about maintaining international peace and security?",

    # Unrelated questions
    "What is the capital city of Japan?",
    "How does a convolutional neural network work?",
    "What is the difference between supervised and unsupervised learning?"
] 

In [23]:
def test_retrieval_questions( questions, document_embeddings, documents, metadatas, k=5, threshold=0.70 ):
    
    for i, query in enumerate(questions, 1):

        print("\n" + "=" * 80)
        print(f"Question {i}: {query}")
        print("=" * 80)

        # 1. Create query embedding
        query_embedding = embed_query(query)

        # 2. Retrieve chunks
        results = get_retrieval_results(
            query=query,
            query_embedding=query_embedding,
            document_embeddings=document_embeddings,
            documents=documents,
            metadatas=metadatas,
            k=k,
            threshold=threshold
        )

        # 3. No results
        if not results:
            print("Not related to the document.")
            continue

        # 4. Chapter query
        if str(results[0]["similarity"]).lower() == "chapter":

            print("Retrieval Type: Chapter")
            print(f"Number of chunks: {len(results)}")

            for j, result in enumerate(results, 1):

                print(f"\n--- Chunk {j} ---")
                print(f"Metadata: {result['metadata']}")
                print(f"Text:\n{result['text']}")

        # 5. Article query
        elif str(results[0]["similarity"]).lower() == "article":

            print("Retrieval Type: Article")

            result = results[0]

            print(f"Metadata: {result['metadata']}")
            print(f"Text:\n{result['text']}")

        # 6. Normal semantic question
        else:

            print("Retrieval Type: Semantic")

            # If multiple chunks were retrieved → use reranker
            if len(results) > 1:

                best_chunk = rerank_results(
                    query=query,
                    results=results
                )

            # If only one chunk was retrieved → no need for reranker
            else:

                best_chunk = results[0]

            print("\n--- Best Chunk ---")
            print(f"Similarity: {best_chunk['similarity']:.4f}")
            
            if "reranker_score" in best_chunk:
                print(f"Reranker Score: {best_chunk['reranker_score']:.4f}")
            
            print(f"Metadata: {best_chunk['metadata']}")
            print(f"Text:\n{best_chunk['text']}")

In [24]:
test_retrieval_questions(
    questions=test_questions,
    document_embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    k=5,
    threshold=0.70
)


Question 1: What does Chapter I of the UN Charter say?
Retrieval Type: Chapter
Number of chunks: 2

--- Chunk 1 ---
Metadata: {'chapter': 'CHAPTER I : PURPOSES AND PRINCIPLES', 'page': 2, 'article': 'Article 1'}
Text:
The Purposes of the United Nations are: 1. To maintain international peace and security, and to that end: to take effective collective measures for the prevention and removal of threats to the peace, and for the suppression of acts of aggression or other breaches of the peace, and to bring about by peaceful means, and in conformity with the principles of justice and international law, adjustment or settlement of international disputes or situations which might lead to a breach of the peace; 2. To develop friendly relations among nations based on respect for the principle of equal rights and self-determination of peoples, and to take other appropriate measures to strengthen universal peace; 3. To achieve international co-operation in solving international problems of an e

In [25]:
def format_context_to_string(retrieval_results):
    # لو مفيش نتائج رجعت
    if not retrieval_results:
        return "No relevant context found."

    context_string = ""
    
    for idx, result in enumerate(retrieval_results, start=1):
        text = result.get("text", "").strip()
        metadata = result.get("metadata", {})
        
        # استخراج البيانات من الـ metadata عشان الـ LLM يعرف يوثق الإجابة
        chapter = metadata.get("chapter", "Unknown Chapter")
        article = metadata.get("article", "Unknown Article")
        
        # تنسيق النص لكل Chunk
        context_string += f"--- Source {idx} ({chapter}, {article}) ---\n"
        context_string += f"The exact text of this article is:\n{text}\n\n"
        
    return context_string.strip() 

In [26]:
import ollama # تأكد من عمل import لمكتبة ollama لو هتستخدمها مباشرة هنا

def test_retrieval_questions_ans(questions, document_embeddings, documents, metadatas, k=5, threshold=0.70):
    
    # لستة عشان نجمع فيها النتائج النهائية لجدول التقييم (Evaluation Table)
    evaluation_results = []
    
    for i, query in enumerate(questions, 1):

        print("\n" + "=" * 80)
        print(f"Question {i}: {query}")
        print("=" * 80)

        # 1. Create query embedding
        query_embedding = embed_query(query)

        # 2. Retrieve chunks
        results = get_retrieval_results(
            query=query,
            query_embedding=query_embedding,
            document_embeddings=document_embeddings,
            documents=documents,
            metadatas=metadatas,
            k=k,
            threshold=threshold
        )

        # 3. No results
        if not results:
            print("Not related to the document.")
            evaluation_results.append({
                "Question": query,
                "Context": "None",
                "LLM Answer": "I don't know (Out of scope).",
                "Status": "Passed (Correctly rejected)"
            })
            continue

        # متغير لتخزين الـ Chunks النهائية اللي هتتبعت للـ Prompt
        final_chunks_to_use = []

        # 4. Chapter query
        if str(results[0]["similarity"]).lower() == "chapter":
            print(f"Retrieval Type: Chapter (Number of chunks: {len(results)})")
            final_chunks_to_use = results

        # 5. Article query
        elif str(results[0]["similarity"]).lower() == "article":
            print("Retrieval Type: Article")
            final_chunks_to_use = [results[0]]

        # 6. Normal semantic question
        else:
            print("Retrieval Type: Semantic")
            if len(results) > 1:
                best_chunk = rerank_results(query=query, results=results)
            else:
                best_chunk = results[0]
            
            print(f"Similarity: {best_chunk['similarity']:.4f}")
            final_chunks_to_use = [best_chunk]

        # =====================================================
        # 7. تحويل النتائج إلى String جاهز للبرومبت
        # =====================================================
        context_string = format_context_to_string(final_chunks_to_use)

        # =====================================================
        # 8. بناء الـ Prompt وطلب الإجابة من الـ LLM
        # =====================================================
        prompt = f"""
        You are a helpful and accurate assistant. 
        Answer the user's question based ONLY on the following context.
        If the context does not contain the answer, say "I don't know".
        Always cite your sources using the chapter and article provided in the context (e.g., "[Page 2 ,Chapter I, Article 1]").

        Context:
        {context_string}

        Question: {query}
        Answer:
        """
        
        # استدعاء الـ LLM (تأكد إن اسم الموديل مطابق للي شغال عندك على Ollama)
        response = ollama.chat(model='llama3', messages=[
            {'role': 'user', 'content': prompt}
        ])
        
        llm_answer = response['message']['content']
        
        # طباعة الإجابة النهائية
        print("\n--- LLM Answer ---")
        print(llm_answer)
        
        # تخزين البيانات عشان تقدر تعمل بيها جدول التقييم في النهاية (Requirement 2.6)
        evaluation_results.append({
            "Question": query,
            "Context Source": context_string,
            "LLM Answer": llm_answer,
            "Correct/Grounded": "True" # ل
        })
        
    return evaluation_results

In [27]:
results_data = test_retrieval_questions_ans(
    questions=test_questions,
    document_embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    k=5,
    threshold=0.70
)


Question 1: What does Chapter I of the UN Charter say?
Retrieval Type: Chapter (Number of chunks: 2)

--- LLM Answer ---
According to the provided context, Chapter I of the UN Charter discusses the Purposes and Principles of the United Nations. Specifically, it mentions the purposes of the UN, which are to maintain international peace and security, develop friendly relations among nations, achieve international cooperation, and be a center for harmonizing the actions of nations. Additionally, it outlines the principles of the organization, including the sovereign equality of its members, the settlement of international disputes by peaceful means, and the refraining from the threat or use of force against any state.

Question 2: What is covered in Chapter V of the UN Charter?
Retrieval Type: Chapter (Number of chunks: 10)

--- LLM Answer ---
According to the provided context, Chapter V of the UN Charter covers the Security Council. Specifically, it deals with the composition, powers, a

In [28]:
import pandas as pd

df_evaluation = pd.DataFrame(results_data)

# 3. اعرض الجدول بشكل منسق
display(df_evaluation)

,Question,Context Source,LLM Answer,Correct/Grounded,Context,Status
0,What does Chapter I of the UN Charter say?,--- Source 1 (CHAPTER I : PURPOSES AND PRINCIP...,"According to the provided context, Chapter I o...",True,NaN,NaN
1,What is covered in Chapter V of the UN Charter?,--- Source 1 (CHAPTER V : THE SECURITY COUNCIL...,"According to the provided context, Chapter V o...",True,NaN,NaN
2,What does Article 46 say?,--- Source 1 (CHAPTER VII : ACTION WITH RESPEC...,"According to the provided context, Article 46 ...",True,NaN,NaN
3,What are the functions of the General Assembly...,--- Source 1 (CHAPTER IV : THE GENERAL ASSEMBL...,According to Article 10 of CHAPTER IV: THE GEN...,True,NaN,NaN
4,What are the purposes of the United Nations?,--- Source 1 (CHAPTER I : PURPOSES AND PRINCIP...,"According to the provided context, the purpose...",True,NaN,NaN
5,What principles should the United Nations and ...,--- Source 1 (CHAPTER I : PURPOSES AND PRINCIP...,According to Article 2 of Chapter I of the sou...,True,NaN,NaN
6,What does the UN Charter say about maintaining...,--- Source 1 (CHAPTER VII : ACTION WITH RESPEC...,"According to the context, the UN Charter state...",True,NaN,NaN
7,What is the capital city of Japan?,NaN,I don't know (Out of scope).,NaN,None,Passed (Correctly rejected)
8,How does a convolutional neural network work?,NaN,I don't know (Out of scope).,NaN,None,Passed (Correctly rejected)
9,What is the difference between supervised and ...,NaN,I don't know (Out of scope).,NaN,None,Passed (Correctly rejected)


In [29]:
df_evaluation = pd.DataFrame(results_data)
# مسح عمودي Context و Status
df_evaluation.drop(columns=['Context', 'Status'], inplace=True, errors='ignore')
display(df_evaluation)

,Question,Context Source,LLM Answer,Correct/Grounded
0,What does Chapter I of the UN Charter say?,--- Source 1 (CHAPTER I : PURPOSES AND PRINCIP...,"According to the provided context, Chapter I o...",True
1,What is covered in Chapter V of the UN Charter?,--- Source 1 (CHAPTER V : THE SECURITY COUNCIL...,"According to the provided context, Chapter V o...",True
2,What does Article 46 say?,--- Source 1 (CHAPTER VII : ACTION WITH RESPEC...,"According to the provided context, Article 46 ...",True
3,What are the functions of the General Assembly...,--- Source 1 (CHAPTER IV : THE GENERAL ASSEMBL...,According to Article 10 of CHAPTER IV: THE GEN...,True
4,What are the purposes of the United Nations?,--- Source 1 (CHAPTER I : PURPOSES AND PRINCIP...,"According to the provided context, the purpose...",True
5,What principles should the United Nations and ...,--- Source 1 (CHAPTER I : PURPOSES AND PRINCIP...,According to Article 2 of Chapter I of the sou...,True
6,What does the UN Charter say about maintaining...,--- Source 1 (CHAPTER VII : ACTION WITH RESPEC...,"According to the context, the UN Charter state...",True
7,What is the capital city of Japan?,NaN,I don't know (Out of scope).,NaN
8,How does a convolutional neural network work?,NaN,I don't know (Out of scope).,NaN
9,What is the difference between supervised and ...,NaN,I don't know (Out of scope).,NaN


In [30]:
json_result = df_evaluation.to_json(orient='records', force_ascii=False, indent=4)
print(json_result)

# 2. لو عايز تحفظ الجدول ده في ملف JSON على الجهاز بتاعك مباشرة 
df_evaluation.to_json('evaluation_results.json', orient='records', force_ascii=False, indent=4)

[
    {
        "Question":"What does Chapter I of the UN Charter say?",
        "Context Source":"--- Source 1 (CHAPTER I : PURPOSES AND PRINCIPLES, Article 1) ---\nThe exact text of this article is:\nThe Purposes of the United Nations are: 1. To maintain international peace and security, and to that end: to take effective collective measures for the prevention and removal of threats to the peace, and for the suppression of acts of aggression or other breaches of the peace, and to bring about by peaceful means, and in conformity with the principles of justice and international law, adjustment or settlement of international disputes or situations which might lead to a breach of the peace; 2. To develop friendly relations among nations based on respect for the principle of equal rights and self-determination of peoples, and to take other appropriate measures to strengthen universal peace; 3. To achieve international co-operation in solving international problems of an economic, social, 

In [31]:
import json
import pandas as pd
import ollama # أو المكتبة اللي بتستخدمها للاتصال بـ LLM

# بافتراض إن df_evaluation هو الجدول اللي طلع معاك في الخطوة اللي فاتت
# 1. اختيار أول 3 أعمدة فقط وتحويلهم لـ JSON

json_data = json_result

# 2. بناء الـ Prompt الخاص بالحكم (LLM-as-a-Judge)
judge_prompt = f"""
You are an expert AI evaluator evaluating a RAG system.

You will receive a JSON array containing multiple independent Q&A evaluation items.

IMPORTANT:
- Each JSON object represents ONE independent evaluation case.
- Evaluate EACH JSON object separately.
- Do NOT combine information from different JSON objects.
- Do NOT use the Context Source from one object to evaluate another object.
- For every object, evaluate ONLY its own:
  "Question"
  "Context Source"
  "LLM Answer"

For each individual item, determine whether the LLM Answer correctly answers
the Question and is strictly grounded in that item's Context Source.

Evaluation Rules:

1. If the item's "Context Source" is "None":
   - If the LLM rejected the question because there was no available context,
     mark:
     "Passed (Correctly rejected)"
   - If the LLM provides an answer despite having no context,
     mark:
     "No (Hallucinated/Incorrect)"

2. If the item's "Context Source" is available:
   - If the LLM Answer correctly answers the Question AND is fully supported
     by the Context Source, mark:
     "Yes (Grounded)"
   - If the answer contains unsupported information, mark:
     "No (Hallucinated/Incorrect)"
   - If the answer contradicts the Context Source, mark:
     "No (Hallucinated/Incorrect)"
   - If the answer does not answer the Question, mark:
     "No (Hallucinated/Incorrect)"

Important:
- Evaluate each JSON object independently.
- Do not use outside knowledge.
- Do not use information from other JSON objects.
- A fact being true in the real world does NOT make it grounded if it is not
  supported by that item's Context Source.
- Paraphrasing or summarizing the Context Source is allowed.
- The order of the output objects MUST be exactly the same as the input.
- Return exactly ONE evaluation for EACH input JSON object.

Output requirements:
- Return ONLY a valid JSON array.
- The output array must contain the same number of objects as the input array.
- Preserve all original fields exactly.
- Add exactly one new field named "Correct/Grounded".
- The value must be exactly one of:

  "Yes (Grounded)"
  "No (Hallucinated/Incorrect)"
  "Passed (Correctly rejected)"

Do not include explanations.
Do not include markdown.
Do not include any text outside the JSON array.

Input JSON:
{json_data}
"""
# 3. إرسال الطلب لـ Ollama (تقدر تستخدم llama3 أو أي موديل تاني)
response = ollama.chat(model='llama3', messages=[
    {'role': 'user', 'content': judge_prompt}
])

judge_output = response['message']['content']

# 4. تنظيف الإجابة (لو الموديل رجع الـ JSON جوه علامات Markdown) وتحويلها لجدول
try:
    # تنظيف سريع تحسباً لأي نصوص إضافية
    cleaned_output = judge_output.replace('```json', '').replace('```', '').strip()
    
    # تحويل الـ JSON اللي رجع لـ List of Dictionaries
    final_evaluated_data = json.loads(cleaned_output)
    
    # تحويله لـ DataFrame وعرضه
    final_df = pd.DataFrame(final_evaluated_data)
    display(final_df)
    
except json.JSONDecodeError:
    print("Failed to parse JSON from the judge LLM. Here is the raw output:")
    print(judge_output)

Failed to parse JSON from the judge LLM. Here is the raw output:
After evaluating the RAG system, I found that:

**Accuracy:** 6 out of 7 questions were answered correctly, with the only incorrect answer being for the question "What is the capital city of Japan?" (which was correctly identified as being out of scope).

**Groundedness:** All 6 correct answers demonstrate a strong understanding of the context and are well-supported by the provided text.

**Reasoning:** The answers demonstrate a clear and logical understanding of the material, with effective use of the provided context to support the answers.

**Transfer learning:** The RAG system's ability to generalize to new, unseen questions is impressive, with all 6 correct answers demonstrating the ability to transfer learning to new contexts.

**Out-of-scope detection:** The RAG system correctly identified the questions "What is the capital city of Japan?", "How does a convolutional neural network work?", and "What is the differenc

In [32]:
# طباعة قيمة Context Source للصف الأول (Index 0)
print(df_evaluation.loc[2, 'Context Source'])

--- Source 1 (CHAPTER VII : ACTION WITH RESPECT TO THREATS TO THE PEACE, BREACHES OF THE PEACE, AND ACTS OF AGGRESSION, Article 46) ---
The exact text of this article is:
Plans for the application of armed force shall be made by the Security Council with the assistance of the Military Staff Committee.
